In [1]:
# Configuration and File Paths
import sys
import time
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from config import get_config, validate_config, setup_directories, print_config_summary
from src.simulation import simulate_asset_damage_recovery_access_optimized
from src.data_loader import load_hazard_maps, load_electricity_assets
import src.grid_based_accessibility_hex as grid_hex

# #TODO: import and run sensitivity analysis


In [2]:
# Get configuration (currently from config.py)
config = get_config()

# Validate and setup
is_valid, missing_dirs = validate_config(config)
setup_directories(config)

# Print summary
print_config_summary(config)

# Handle validation results
if missing_dirs:
    print(f"\nWarning: Missing directories: {missing_dirs}")
else:
    print("\nAll data directories found successfully!")

# Set the hazard extraction method constant
HAZARD_EXTRACTION_METHOD = config['analysis_config']['hazard_extraction_method']
print(f"Hazard extraction method set to: {HAZARD_EXTRACTION_METHOD}")



Configuration Summary
Root directory: c:\repos\powerpath
Assets data: c:\repos\powerpath\data\electricity
Hazard data: N:\Projects\11209000\11209175\B. Measurements and calculations\Data\timeseries_data\reprojected
Interim directory: c:\repos\powerpath\data\interim\interim_reprojected
Output directory: c:\repos\powerpath\data\output\output_reprojected

Simulation Configuration:
  number_repair_crews: 10
  repair_crew_assignment_method: islands
  flood_threshold: 0.2
  verbose: True

Recovery Parameters:
  repair_time_coefficients: [702.72, 3.14, 1.9891]
  damage_ratio_coefficients: (0.0468, 0.0077)
  time_step_hours: 1
  damage_threshold: 0.01
  repair_threshold: 2.0

Analysis Configuration:
  hazard_extraction_method: max
  max_simulation_days: None
  cache_enabled: True
  performance_monitoring: False

All data directories found successfully!
Hazard extraction method set to: max


In [3]:
# Load data
print("Loading electricity assets...")
gdf_assets = load_electricity_assets(config['electricity_dir'])

print("\nLoading hazard maps...")
hazard_maps = load_hazard_maps(config['hazard_dir'], max_days=10)  
print("Data loading completed!")
print(f"\nHazard maps loaded:")
for i, hm in enumerate(hazard_maps):
    if i < 3:
        print(f"  -{hm}")
    elif i == 3:
        print(f"  and {len(hazard_maps) - 3} more")
        break




Loading electricity assets...
Loading electricity assets from ls_stations_clipped.shp
Loaded 603 ls assets
Loading electricity assets from msls_stations_clipped.shp
Loaded 1258 msls assets
Combined total: 1861 electricity assets
Asset types: {'msls': 1258, 'ls': 603}

Loading hazard maps...
Found 9 hazard map files
Data loading completed!

Hazard maps loaded:
  -N:\Projects\11209000\11209175\B. Measurements and calculations\Data\timeseries_data\reprojected\0_blank_map.tif
  -N:\Projects\11209000\11209175\B. Measurements and calculations\Data\timeseries_data\reprojected\20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-03_scale_5_wgs84.tif
  -N:\Projects\11209000\11209175\B. Measurements and calculations\Data\timeseries_data\reprojected\20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-04_scale_5_wgs84.tif
  and 6 more


In [10]:
# Configure simulation parameters from config
simulation_params = {
    'flood_threshold': config['simulation_config']['flood_threshold'],
    'number_repair_crews': config['simulation_config']['number_repair_crews'],
    'repair_crew_assignment_method': config['simulation_config']['repair_crew_assignment_method'],
    'verbose': config['simulation_config']['verbose'],
    'damage_ratio_coefficients': config['recovery_parameters']['damage_ratio_coefficients'],
    'repair_time_coefficients': config['recovery_parameters']['repair_time_coefficients'],
    'damage_threshold': config['recovery_parameters']['damage_threshold'],
    'repair_threshold': config['recovery_parameters']['repair_threshold'],
    'config': config  # Pass entire config for directory management
}

accessibility_model = grid_hex.accessibility_model
simulation_params['accessibility_model'] = accessibility_model



print(f"\nSimulation configuration:")
for key, value in simulation_params.items():
    if key != 'config':  # Don't print the entire config
        print(f"  {key}: {value}")

print(f"\nDirectory structure:")
print(f"  Interim: {config['interim_dir']}")
print(f"  Output: {config['output_dir']}")
print(f"  Cache will be organized by hazard directory: {Path(config['hazard_dir']).name}")



Simulation configuration:
  flood_threshold: 0.2
  number_repair_crews: 10
  repair_crew_assignment_method: islands
  verbose: True
  damage_ratio_coefficients: (0.0468, 0.0077)
  repair_time_coefficients: [702.72, 3.14, 1.9891]
  damage_threshold: 0.01
  repair_threshold: 2.0
  accessibility_model: <function accessibility_model at 0x000001D800024860>

Directory structure:
  Interim: c:\repos\powerpath\data\interim\interim_reprojected
  Output: c:\repos\powerpath\data\output\output_reprojected
  Cache will be organized by hazard directory: reprojected


In [11]:
# simulation_params['verbose'] = False  # Set verbose to False for cleaner output

execution_id = int(time.time())
print(f"Starting simulation execution {execution_id}")

# Run the  simulation
results_df, final_state = simulate_asset_damage_recovery_access_optimized(
    gdf_assets=gdf_assets,
    hazard_maps=hazard_maps,
    number_repair_crews=simulation_params['number_repair_crews'],
    repair_crew_assignment_method=simulation_params['repair_crew_assignment_method'],
    flood_threshold=simulation_params['flood_threshold'],
    recovery_parameters=config['recovery_parameters'],
    root_dir=config['root_dir'],
    verbose=simulation_params['verbose']
)

print(f"✅ Completed simulation execution {execution_id}")

Starting simulation execution 1754557539
Using hazard directory for cache naming: reprojected
Loading optimization caches...
Loaded accessibility cache: 18 entries from c:\repos\powerpath\data\interim\accessibility_cache_reprojected.pkl
Loaded overlap cache: 8 entries from c:\repos\powerpath\data\interim\overlap_cache_reprojected.pkl
Loaded hazard extraction cache: 9 entries from c:\repos\powerpath\data\interim\cache\hazard_extraction_cache_reprojected.pkl
Pre-computing island assignments...
Pre-computing island assignments for 9 hazard maps and 3 thresholds...
Loaded island cache: 27 entries from c:\repos\powerpath\data\interim\cache\island_cache_reprojected.pkl
Saved island cache: 27 entries to c:\repos\powerpath\data\interim\cache\island_cache_reprojected.pkl
Island assignment pre-computation complete:
  Total combinations: 27
  Newly computed: 0
  Skipped (cached): 27
Initializing grid-based accessibility analysis...
Island-based method 'islands' will be used

=== Processing timest